# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maahilkureshi-design/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The queue ranks pages by the model's probability that their impressions will fall to half or less of their client's trend over the next 30 days. Next to each page it shows reason codes (watch-list conditions the page meets) and a suggested review action. Reason codes describe the page; they are not causes and not exact model attributions.

| Reason code | What it means | What a reviewer checks first |
|---|---|---|
| `VOLATILE_DAILY_IMPRESSIONS` | Daily impressions swing far more than for most pages | Whether recent weeks look unusual against the page's own history |
| `INTERMITTENT_VISIBILITY` | The page showed up on few of the last 90 days | Whether the page is really indexed and eligible to rank |
| `FALLING_VS_CLIENT` | Last 30 days fell about a third or more against the client's trend | Whether the fall is real or a return to normal after a spike |
| `DEEP_POSITION` | Average position is 30 or worse | Whether the page can realistically move up |
| `POSITION_SLIPPING` | Average position got 3 places or more worse | What changed on the page or in competing pages |
| `NO_CLICKS_LAST_30D` | No clicks in the last 30 days | Whether the title and snippet match what searchers want |

Suggested actions: `review now: improve or refresh` (high risk and 1,000+ impressions), `review: improve or monitor` (high risk), `review for merge or prune` (high risk, low volume, intermittent and deep), `monitor: recent rise may fade` (high risk but the page is already growing, so a refresh does not fit), `protect` (page is growing and not high risk), `monitor` (everything else). A page with no reason code shows `no watch-list condition met`: the model ranked it high on the combination of its features. Pages tagged `recent spike` are not scored; the advice is to expect a pullback and monitor.

In [1]:
import os, json, shutil
import numpy as np
import pandas as pd
import duckdb
from sklearn.metrics import average_precision_score

# The saved snapshot files and shared outputs live in Google Drive (MyDrive/flyrank_capstone),
# so this notebook can run in its own session after capstone.ipynb has finished.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT, ON_COLAB = '/content/drive/MyDrive/flyrank_capstone', True
except ImportError:
    DRIVE_ROOT, ON_COLAB = '.', False
CACHE_DIR, EVAL_DIR, OUT_DIR = 'snap_cache', 'eval_cache', f'{DRIVE_ROOT}/outputs'
if ON_COLAB:
    assert os.path.exists(f'{DRIVE_ROOT}/snap_cache'), 'Not found in Google Drive: run capstone.ipynb first, then its last cell (copy to Drive).'
    shutil.copytree(f'{DRIVE_ROOT}/snap_cache', CACHE_DIR, dirs_exist_ok=True)      # local copy: faster than reading from Drive
    if os.path.exists(f'{DRIVE_ROOT}/eval_cache'):
        shutil.copytree(f'{DRIVE_ROOT}/eval_cache', EVAL_DIR, dirs_exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
n_files = len([f for f in os.listdir(CACHE_DIR) if f.startswith('snap_') and f.endswith('.parquet')]) if os.path.exists(CACHE_DIR) else 0
assert n_files == 12, f'{n_files} snapshot files found, expected 12: run capstone.ipynb and its last cell (copy to Drive) first.'

LABEL_DAYS = 30
K_LIST = [20, 50]
MIN_CLIENT_PAGES_MODEL = 200
SPIKE_RULE = "imp_last30 >= 2 * GREATEST(imp_prev30, imp_prev60)"
UNIVERSE = f"n_elig >= {MIN_CLIENT_PAGES_MODEL} AND NOT ({SPIKE_RULE})"
PAST_COLS = ['imp_90', 'clicks_90', 'ctr_90', 'ctr_last30', 'pos_90', 'pos_last30', 'pos_prev30', 'pos_change',
             'imp_last30', 'imp_prev30', 'imp_prev60', 'active_days_90', 'cv_90',
             'log_mom', 'log_mom_rel', 'log_mom_prev', 'log_mom_prev_rel', 'page_age_days']
STATIC_COLS = ['word_count', 'char_count', 'content_type', 'main_intent', 'competition_level', 'competition',
               'search_volume', 'cpc', 'backlinks', 'category_count', 'keyword_token_count', 'url_char_count']
mcon = duckdb.connect()

def _clean(d):
    d['snap_t'] = pd.to_datetime(d['snap_t'])
    for c in PAST_COLS:
        if pd.api.types.is_timedelta64_dtype(d[c]):
            d[c] = d[c].dt.days
    return d

def load_train():
    cols = ', '.join(['snap_t', 'client_hash_id', 'y'] + PAST_COLS)
    return _clean(mcon.sql(f"SELECT {cols} FROM read_parquet('{CACHE_DIR}/snap_*.parquet') WHERE {UNIVERSE}").df())

def load_eval(name):
    path = f'{EVAL_DIR}/eval_{name}.parquet'
    assert os.path.exists(path), f'{path} is missing: run the build cells of capstone.ipynb first'
    cols = ', '.join(['snap_t', 'client_hash_id', 'y'] + PAST_COLS)
    return _clean(mcon.sql(f"SELECT {cols} FROM read_parquet('{path}') WHERE {UNIVERSE}").df())

def eval_rule(df, score_col, ks=K_LIST):
    rows = []
    for t, g in df.groupby('snap_t'):
        g = g.sort_values(score_col, ascending=False, kind='mergesort')
        base = g['y'].mean()
        row = {'snap_t': str(t)[:10], 'n': len(g), 'base_rate': round(base, 3)}
        for k in ks:
            p = g['y'].head(k).mean()
            row[f'p@{k}'] = round(p, 3)
            row[f'lift@{k}'] = round(p / base, 2) if base > 0 else np.nan
        ap = average_precision_score(g['y'], g[score_col]) if g['y'].sum() > 0 else np.nan
        row['ap'] = round(ap, 3)
        row['ap_lift'] = round(ap / base, 2) if base > 0 else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

GLOSSARY = pd.DataFrame([
    ('VOLATILE_DAILY_IMPRESSIONS', 'volatility in the top 10% of scored pages'),
    ('INTERMITTENT_VISIBILITY', 'active days in the bottom 10% of scored pages'),
    ('FALLING_VS_CLIENT', 'log_mom_rel <= -0.4'),
    ('DEEP_POSITION', 'average position over the last 30 days >= 30'),
    ('POSITION_SLIPPING', 'position worse by 3 or more versus the previous 30 days'),
    ('NO_CLICKS_LAST_30D', 'click-through rate of zero in the last 30 days'),
], columns=['reason_code', 'rule'])
print(GLOSSARY.to_string(index=False))

queue_path = os.path.join(OUT_DIR, 'ranked_queue_sample.csv')
if os.path.exists(queue_path):
    queue = pd.read_csv(queue_path)
    print(f"\nFinal queue sample: {len(queue)} rows")
    print(queue.head(10).to_string(index=False))
    print('\nSuggested actions in the sample:')
    print(queue['action'].value_counts().to_string())
else:
    print('ranked_queue_sample.csv not found in', OUT_DIR, ': run the final-queue cell of capstone.ipynb, then its last cell (copy to Drive).')

Mounted at /content/drive
               reason_code                                                    rule
VOLATILE_DAILY_IMPRESSIONS               volatility in the top 10% of scored pages
   INTERMITTENT_VISIBILITY           active days in the bottom 10% of scored pages
         FALLING_VS_CLIENT                                     log_mom_rel <= -0.4
             DEEP_POSITION            average position over the last 30 days >= 30
         POSITION_SLIPPING position worse by 3 or more versus the previous 30 days
        NO_CLICKS_LAST_30D          click-through rate of zero in the last 30 days

Final queue sample: 50 rows
 rank  client_rank  p_decline  imp_last30  imp_prev30     tag                                                           reason_codes                        action
    1            5     0.8356       391.0       810.0  stable              INTERMITTENT_VISIBILITY, DEEP_POSITION, POSITION_SLIPPING     review for merge or prune
    2            5     0.8227       18

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who uses it.** A content strategist or editor choosing which pages to review next cycle, out of thousands, when there is time for roughly 20 to 50.

**What it is for.** Deciding where to look first. It is decision-support: it ranks pages by observed risk of a relative decline and shows why a page was flagged.

**Where it is valid.** Pages that are published, at least 90 days old, with at least 100 impressions in the last 30 days, from clients with at least 200 such pages, and not new or spiking. The score looks one month ahead. Pages outside those conditions are not scored.

**Where it stops.** It was validated on two out-of-time months and its edge over simple rules varies by month and by page volume. It does not show why a page moves, that an edit would help, or how Google ranks pages. The base rate of decline changed a lot between months, so scores are for ranking, not for forecasting how many pages will fall.

In [2]:
INTENDED_USE = {
    'user': 'content strategist or editor',
    'decision': 'which pages to review first next cycle (20 to 50 pages)',
    'horizon': 'the next 30 days after the snapshot date',
    'universe': 'published, 90+ days old, 100+ impressions in the last 30 days, client with 200+ such pages, not new or spiking',
    'output': 'rank, probability of relative decline, reason codes, suggested review action',
    'status': 'decision-support; not causal, not a forecast of counts',
}
for k, v in INTENDED_USE.items():
    print(f'{k:>9}: {v}')

     user: content strategist or editor
 decision: which pages to review first next cycle (20 to 50 pages)
  horizon: the next 30 days after the snapshot date
 universe: published, 90+ days old, 100+ impressions in the last 30 days, client with 200+ such pages, not new or spiking
   output: rank, probability of relative decline, reason codes, suggested review action
   status: decision-support; not causal, not a forecast of counts


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**What a person must check before acting**

1. Open the page's last 90 days. Is the fall real, or a return to normal after a spike?
2. Check for events that move a whole site (seasonality, an outage, a tracking change) before blaming the page.
3. For low-volume pages (under 1,000 impressions), treat the ranking as a hint: counts are noisy.
4. Check search intent and the page's role (is it a page that should exist, or has a better page absorbed it?).

**The no-go list: never automate these**

- Deleting, merging, redirecting or unpublishing a page because it is in the queue.
- Claiming that a refresh caused a recovery. No experiment was run.
- Scoring pages outside the validity conditions, or new or spiking pages.
- Using the probabilities as counts ("X pages will fall").
- Publishing client-level results, names or page identifiers.

In [3]:
REVIEW_CHECKLIST = [
    'last 90 days of the page reviewed by a person',
    'site-wide events ruled out (seasonality, outage, tracking change)',
    'volume noted: under 1,000 impressions means treat the rank as a hint',
    'intent and the page role checked',
]
NO_GO = [
    'automatic delete, merge, redirect or unpublish',
    'causal claims about refreshes',
    'scoring outside the validity conditions',
    'treating probabilities as counts',
    'publishing client-level identifiers',
]
print('Before acting:'); [print(' -', x) for x in REVIEW_CHECKLIST]
print('Never automate:'); [print(' -', x) for x in NO_GO]

Before acting:
 - last 90 days of the page reviewed by a person
 - site-wide events ruled out (seasonality, outage, tracking change)
 - volume noted: under 1,000 impressions means treat the rank as a hint
 - intent and the page role checked
Never automate:
 - automatic delete, merge, redirect or unpublish
 - causal claims about refreshes
 - scoring outside the validity conditions
 - treating probabilities as counts
 - publishing client-level identifiers


[None, None, None, None, None]

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**What would tell me the recommendations went stale** (checked each month once the next 30 days are known):

- **Base-rate drift.** The share of scored pages that actually declined moves outside half to double its recent median. This happened in the study months: the base rate went from about 4% to 8% through the winter to about 17% to 22% in spring.
- **Lost skill.** AP lift (average precision divided by the base rate) stays below 1.3, the level of the strongest simple rule, for two months in a row.
- **Weak top of the list.** Lift at the top 50 stays below 2.0 for two months in a row.
- **New clients.** More than 30% of scored rows come from clients that were not in the training data.

Any trigger means: re-check the label behaviour, retrain on the latest snapshots, and re-run the leakage test before the queue is used again.

In [4]:
# Monthly results reported in capstone.ipynb (walk-forward folds, validation, sealed test)
monthly = pd.DataFrame([
    ('2025-10-31', 'train fold', 0.048, 2.47, 5.44), ('2025-11-30', 'train fold', 0.040, 2.32, 3.04),
    ('2025-12-31', 'train fold', 0.073, 2.59, 8.77), ('2026-01-31', 'train fold', 0.047, 2.40, 11.52),
    ('2026-02-28', 'train fold', 0.080, 2.23, 5.53), ('2026-03-31', 'train fold', 0.167, 1.68, 3.23),
    ('2026-04-30', 'validation', 0.169, 1.84, 2.85), ('2026-05-31', 'sealed test', 0.221, 1.52, 2.80),
], columns=['snapshot', 'period', 'base_rate', 'ap_lift', 'lift_at_50'])

prior_median = monthly['base_rate'].shift(1).expanding(min_periods=3).median()
monthly['T1_base_rate_drift'] = (monthly['base_rate'] > 2 * prior_median) | (monthly['base_rate'] < 0.5 * prior_median)
monthly['low_skill'] = monthly['ap_lift'] < 1.3
monthly['T2_lost_skill'] = monthly['low_skill'] & monthly['low_skill'].shift(1, fill_value=False)
monthly['weak_top'] = monthly['lift_at_50'] < 2.0
monthly['T3_weak_top'] = monthly['weak_top'] & monthly['weak_top'].shift(1, fill_value=False)
print(monthly[['snapshot', 'period', 'base_rate', 'ap_lift', 'lift_at_50', 'T1_base_rate_drift', 'T2_lost_skill', 'T3_weak_top']].to_string(index=False))
print('\nTriggers fired in:', monthly.loc[monthly[['T1_base_rate_drift', 'T2_lost_skill', 'T3_weak_top']].any(axis=1), 'snapshot'].tolist())

  snapshot      period  base_rate  ap_lift  lift_at_50  T1_base_rate_drift  T2_lost_skill  T3_weak_top
2025-10-31  train fold      0.048     2.47        5.44               False          False        False
2025-11-30  train fold      0.040     2.32        3.04               False          False        False
2025-12-31  train fold      0.073     2.59        8.77               False          False        False
2026-01-31  train fold      0.047     2.40       11.52               False          False        False
2026-02-28  train fold      0.080     2.23        5.53               False          False        False
2026-03-31  train fold      0.167     1.68        3.23                True          False        False
2026-04-30  validation      0.169     1.84        2.85                True          False        False
2026-05-31 sealed test      0.221     1.52        2.80                True          False        False

Triggers fired in: ['2026-03-31', '2026-04-30', '2026-05-31']


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The paper builds on these files in `work/outputs/`: the ranked queue sample (no identifiers), the reason-code glossary, and the monthly monitoring table.

In [5]:
GLOSSARY.to_csv(os.path.join(OUT_DIR, 'reason_code_glossary.csv'), index=False)
monthly.to_csv(os.path.join(OUT_DIR, 'monthly_monitoring.csv'), index=False)
print('Files in', OUT_DIR + ':')
for f in sorted(os.listdir(OUT_DIR)):
    print(' -', f)

Files in /content/drive/MyDrive/flyrank_capstone/outputs:
 - baseline_action_score.csv
 - fig_distributions.png
 - leakage_test.json
 - monthly_monitoring.csv
 - permutation_importance.csv
 - ranked_queue_high_volume_sample.csv
 - ranked_queue_sample.csv
 - reason_code_glossary.csv
 - signal_verdicts.csv
 - walkforward_summary.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.